In [ ]:
%pip install gradio
%pip install openai
%pip install tiktoken
%pip install langchain duckduckgo-search

In [2]:
import os

# ホームディレクトリの下にあるファイルのパスを作成
file_path = os.path.join(os.path.expanduser("~"), 'dotsecret')

# ファイルを開いて各行を読みます
variables = {}
with open(file_path, 'r') as file:
    for line in file:
        # 空白行を無視します
        if line.strip():
            key, value = line.strip().split('=')
            os.environ[key] = value

In [6]:
import openai
import os
import gradio as gr
from langchain.tools import DuckDuckGoSearchRun

# OpenAI APIキーを設定
openai.api_key = os.getenv("OPENAI_API_KEY")

def duckduckgo_search(query):
    search = DuckDuckGoSearchRun()
    search_result = search.run(query)
    return search_result

def generate_answer_with_chatgpt(search_result, query):
    instruction = f"Based on the following search result, answer the user's query: '{query}'.\n\nSearch Result:\n{search_result}"
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": instruction},
        ]
    )
    answer = response['choices'][0]['message']['content']
    return answer

def openai_chat(messages, history):
    user_message = messages
    search_result = duckduckgo_search(user_message)
    answer = generate_answer_with_chatgpt(search_result, user_message)
    return answer

with gr.Blocks() as app:
    with gr.Tab("Chat"):
        gr.ChatInterface(fn=openai_chat)

if __name__ == "__main__":
    app.launch()


Running on local URL:  http://127.0.0.1:7900

To create a public link, set `share=True` in `launch()`.
